# 🎬 Bollywood Box Office Predictor
## Notebook 5: ML Prediction Model + SHAP Explainability

**Goal:**
- Load final feature matrix from Notebook 4
- Train Random Forest and XGBoost classifiers
- Use Leave-One-Out Cross Validation (LOOCV) for small dataset
- Evaluate with confusion matrix, accuracy, F1 score
- Apply SHAP for model explainability
- Understand WHY each movie was predicted as hit/flop
- Save predictions

### Why LOOCV?
```
With only 12 movies:
   Standard 80/20 split = 9 train / 2 test → unreliable
   LOOCV = train on 11, test on 1, repeat 12 times → honest evaluation
   Every movie gets tested exactly once
   Industry standard for small datasets (<50 samples)
```

In [ ]:
# ── CELL 2: INSTALL & IMPORTS ─────────────────────────────────────────────
!pip install xgboost shap scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')
from collections import Counter

from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import LabelEncoder
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics         import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)
import xgboost as xgb

plt.style.use('seaborn-v0_8-darkgrid')
print(f'✅ XGBoost version : {xgb.__version__}')
print(f'✅ SHAP version    : {shap.__version__}')

In [ ]:
# ── CELL 3: MOUNT DRIVE & LOAD DATA ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/Bollywood_Predictor/'
df        = pd.read_csv(f'{SAVE_PATH}final_features.csv')

print(f'✅ Data loaded: {df.shape}')
print(f'\n📊 Label distribution:')
print(df['label'].value_counts())

# Prepare X and y
meta_cols    = ['name','label','label_num','roi_pct','budget_cr','collection_cr']
feature_cols = [c for c in df.columns if c not in meta_cols]

X  = df[feature_cols]
y  = df['label_num']
le = LabelEncoder()
le.fit(df['label'])

print(f'\n📊 Features : {X.shape[1]}')
print(f'📊 Samples  : {X.shape[0]}')
print(f'📊 Classes  : {list(le.classes_)}')

In [ ]:
# ── CELL 4: SMALL DATASET STRATEGY ───────────────────────────────────────

print('📊 Dataset Size Analysis:')
print(f'   Total movies  : {len(df)}')
print(f'   Total features: {X.shape[1]}')
print(f'   Classes       : {list(le.classes_)}')
print(f"""
Strategy: Leave-One-Out Cross Validation (LOOCV)
─────────────────────────────────────────────────
Why not standard 80/20 split?
   • 80/20 of 12 movies = 9 train / 2 test
   • 2 test samples = unreliable accuracy estimate
   • Results change drastically with different split

Why LOOCV?
   • Train on 11 movies, test on 1
   • Repeat 12 times — every movie tested once
   • Maximum available training data each time
   • Gives honest per-movie prediction
   • Industry standard for small datasets

Why balanced class weights?
   • Our dataset is naturally imbalanced
   • 8 Blockbusters, 3 Flops, 1 Average
   • Without balancing → model predicts Blockbuster always
   • Balanced weights penalize minority class errors more
   • This mirrors real industry data distribution
""")

In [ ]:
# ── CELL 5: MODEL 1 — RANDOM FOREST ──────────────────────────────────────

print('⚙️  Training Random Forest with LOOCV...')

rf_model = RandomForestClassifier(
    n_estimators      = 300,
    max_depth         = 4,
    min_samples_split = 2,
    random_state      = 42,
    class_weight      = 'balanced'
)

loo          = LeaveOneOut()
rf_preds     = []
rf_actuals   = []
rf_probs     = []
rf_per_movie = []

for train_idx, test_idx in loo.split(X):
    X_train = X.iloc[train_idx]
    X_test  = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    rf_model.fit(X_train, y_train)
    pred = rf_model.predict(X_test)
    prob = rf_model.predict_proba(X_test)

    rf_preds.append(pred[0])
    rf_actuals.append(y_test.values[0])
    rf_probs.append(prob[0])

    movie_name      = df['name'].iloc[test_idx[0]]
    actual_label    = le.inverse_transform([y_test.values[0]])[0]
    predicted_label = le.inverse_transform([pred[0]])[0]
    correct         = actual_label == predicted_label
    rf_per_movie.append({
        'movie'     : movie_name,
        'actual'    : actual_label,
        'predicted' : predicted_label,
        'correct'   : correct,
        'confidence': max(prob[0])
    })

rf_accuracy = accuracy_score(rf_actuals, rf_preds)
rf_f1       = f1_score(rf_actuals, rf_preds, average='weighted')

print(f'\n✅ Random Forest Results:')
print(f'   Accuracy : {rf_accuracy*100:.1f}%')
print(f'   F1 Score : {rf_f1:.3f}')
print(f'\n📊 Per Movie Predictions:')
print(f'{"Movie":<35} {"Actual":<15} {"Predicted":<15} {"Result"}')
print('─' * 75)
for m in rf_per_movie:
    icon = '✅' if m['correct'] else '❌'
    print(f'{m["movie"]:<35} {m["actual"]:<15} {m["predicted"]:<15} {icon}')

print(f'\n📊 Classification Report:')
print(classification_report(rf_actuals, rf_preds, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(rf_actuals, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Random Forest — Confusion Matrix', fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 6: MODEL 2 — XGBOOST ────────────────────────────────────────────

print('⚙️  Training XGBoost with LOOCV...')

# Calculate class weights manually for XGBoost
class_counts  = Counter(y)
total         = len(y)
class_weights = {cls: total/(len(class_counts)*count)
                 for cls, count in class_counts.items()}

xgb_model = xgb.XGBClassifier(
    n_estimators     = 300,
    max_depth        = 3,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 1,
    random_state     = 42,
    eval_metric      = 'mlogloss',
    verbosity        = 0
)

xgb_preds    = []
xgb_actuals  = []
xgb_probs    = []
xgb_per_movie = []

for train_idx, test_idx in loo.split(X):
    X_train = X.iloc[train_idx]
    X_test  = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    sample_weights = [class_weights[yi] for yi in y_train]
    xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
    pred = xgb_model.predict(X_test)
    prob = xgb_model.predict_proba(X_test)

    xgb_preds.append(pred[0])
    xgb_actuals.append(y_test.values[0])
    xgb_probs.append(prob[0])

    movie_name      = df['name'].iloc[test_idx[0]]
    actual_label    = le.inverse_transform([y_test.values[0]])[0]
    predicted_label = le.inverse_transform([pred[0]])[0]
    correct         = actual_label == predicted_label
    xgb_per_movie.append({
        'movie'     : movie_name,
        'actual'    : actual_label,
        'predicted' : predicted_label,
        'correct'   : correct,
        'confidence': max(prob[0])
    })

xgb_accuracy = accuracy_score(xgb_actuals, xgb_preds)
xgb_f1       = f1_score(xgb_actuals, xgb_preds, average='weighted')

print(f'\n✅ XGBoost Results:')
print(f'   Accuracy : {xgb_accuracy*100:.1f}%')
print(f'   F1 Score : {xgb_f1:.3f}')
print(f'\n📊 Per Movie Predictions:')
print(f'{"Movie":<35} {"Actual":<15} {"Predicted":<15} {"Result"}')
print('─' * 75)
for m in xgb_per_movie:
    icon = '✅' if m['correct'] else '❌'
    print(f'{m["movie"]:<35} {m["actual"]:<15} {m["predicted"]:<15} {icon}')

print(f'\n📊 Classification Report:')
print(classification_report(xgb_actuals, xgb_preds, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(xgb_actuals, xgb_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('XGBoost — Confusion Matrix', fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}xgb_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 7: MODEL COMPARISON ──────────────────────────────────────────────

comparison = pd.DataFrame({
    'Model'   : ['Random Forest', 'XGBoost'],
    'Accuracy': [rf_accuracy*100, xgb_accuracy*100],
    'F1 Score': [rf_f1, xgb_f1]
}).round(3)

print('🏆 Model Comparison:')
print(comparison.to_string(index=False))

best_model_name = 'XGBoost' if xgb_accuracy >= rf_accuracy else 'Random Forest'
best_accuracy   = max(xgb_accuracy, rf_accuracy)
best_preds      = xgb_per_movie if xgb_accuracy >= rf_accuracy else rf_per_movie

print(f'\n🏆 Winner: {best_model_name} ({best_accuracy*100:.1f}% accuracy)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

for ax, metric, vals in zip(
    axes,
    ['Accuracy (%)', 'F1 Score'],
    [[rf_accuracy*100, xgb_accuracy*100], [rf_f1, xgb_f1]]
):
    bars = ax.bar(['Random Forest', 'XGBoost'], vals,
                   color=['#0984E3', '#00B894'], alpha=0.85,
                   edgecolor='white', width=0.5)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 8: SHAP EXPLAINABILITY ───────────────────────────────────────────

print('⚙️  Computing SHAP values...')
print('   Training on full dataset for SHAP analysis')

# Train on full data for SHAP
xgb_model.fit(X, y)

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X)

print('✅ SHAP values computed')

# Global feature importance via SHAP
plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values, X,
    plot_type   = 'bar',
    class_names = list(le.classes_),
    show        = False,
    max_display = 20
)
plt.title('SHAP Feature Importance — All Classes', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Beeswarm for each class
class_names = list(le.classes_)
for i, class_name in enumerate(class_names):
    plt.figure(figsize=(12, 7))
    shap.summary_plot(shap_values[i], X, show=False, max_display=15)
    plt.title(f'SHAP Beeswarm — {class_name} Prediction Drivers',
              fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{SAVE_PATH}shap_beeswarm_{class_name.lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ {class_name} SHAP plot saved')

In [ ]:
# ── CELL 9: PER MOVIE SHAP EXPLANATION ───────────────────────────────────

print(f'🏆 Best Model: {best_model_name} ({best_accuracy*100:.1f}% accuracy)')
print(f'\n📊 Per Movie Analysis with SHAP:\n')
print('─' * 80)

for i, movie_info in enumerate(best_preds):
    movie_name = movie_info['movie']
    actual     = movie_info['actual']
    predicted  = movie_info['predicted']
    correct    = movie_info['correct']
    confidence = movie_info['confidence']
    icon       = '✅' if correct else '❌'

    print(f'{icon} {movie_name}')
    print(f'   Actual    : {actual}')
    print(f'   Predicted : {predicted}')
    print(f'   Confidence: {confidence*100:.1f}%')

    # Top SHAP drivers for this movie
    actual_class_idx = list(le.classes_).index(actual)
    movie_shap       = shap_values[actual_class_idx][i]
    shap_series      = pd.Series(
        np.abs(movie_shap), index=feature_cols
    ).nlargest(5)

    print(f'   Top 5 prediction drivers:')
    for feat, val in shap_series.items():
        feat_idx  = list(feature_cols).index(feat)
        direction = '↑ pushes toward' if movie_shap[feat_idx] > 0 else '↓ pushes against'
        print(f'      {direction} {actual}: {feat} (impact: {val:.4f})')
    print()

In [ ]:
# ── CELL 10: SAVE PREDICTIONS ─────────────────────────────────────────────

predictions_df = pd.DataFrame({
    'movie'           : [m['movie']      for m in best_preds],
    'actual_label'    : [m['actual']     for m in best_preds],
    'predicted_label' : [m['predicted']  for m in best_preds],
    'correct'         : [m['correct']    for m in best_preds],
    'confidence'      : [m['confidence'] for m in best_preds],
    'roi_pct'         : df['roi_pct'].values,
    'budget_cr'       : df['budget_cr'].values,
    'collection_cr'   : df['collection_cr'].values,
    'best_model'      : best_model_name,
})

predictions_df['rf_confidence']  = [max(p) for p in rf_probs]
predictions_df['xgb_confidence'] = [max(p) for p in xgb_probs]

predictions_df.to_csv(f'{SAVE_PATH}predictions.csv', index=False)

print(f'✅ Predictions saved')
print(f'\n🏆 Final Results:')
print(f'   Random Forest Accuracy : {rf_accuracy*100:.1f}%')
print(f'   XGBoost Accuracy       : {xgb_accuracy*100:.1f}%')
print(f'   Best Model             : {best_model_name}')
print(f'   Best Accuracy          : {best_accuracy*100:.1f}%')
print(f'\n📊 Prediction Summary:')
print(f'{"Movie":<35} {"Actual":<15} {"Predicted":<15} {"Confidence":>10} {"Result"}')
print('─' * 85)
for _, row in predictions_df.iterrows():
    icon = '✅' if row['correct'] else '❌'
    print(f'{row["movie"]:<35} {row["actual_label"]:<15} '
          f'{row["predicted_label"]:<15} {row["confidence"]*100:>9.1f}% {icon}')

print(f'\n✅ Notebook 5 complete — proceed to Notebook 6: Insights Dashboard')